In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ f = xW + b $$
$$ \frac{\partial f}{\partial x} = W $$
$$ \frac{\partial f}{\partial W} = x $$
$$ \frac{\partial f}{\partial b} = 1 $$

In [ ]:
import torch.autograd as autograd
import torch.nn as nn
import torch as tr

import import_ipynb
from common import assert_eq # type: ignore

def linear(x: tr.Tensor, W: tr.Tensor, b: tr.Tensor) -> tr.Tensor:
    """Linear function: y = x @ W + b."""

    # (S, out_features) = (S, in_features) @ (in_features, out_features) + (out_features,)
    return tr.addmm(b, x, W)


class LinearFunction(autograd.Function):
    """
    Custom autograd.Function implementing a linear layer:

        y = x @ W + b

    with explicit forward and backward passes.
    """

    @staticmethod
    def forward(ctx, x: tr.Tensor, W: tr.Tensor, b: tr.Tensor) -> tr.Tensor:
        ctx.save_for_backward(x, W)
        z = linear(x, W, b)
        return z

    @staticmethod
    def backward(ctx, dF_df: tr.Tensor) -> tuple[tr.Tensor, tr.Tensor, tr.Tensor]:
        (x, W) = ctx.saved_tensors
        (S, FI, FO) = x.shape[0], x.shape[1], W.shape[1]
       
        # (samples, features) = (samples, output) @ (output, features)
        dF_dx = dF_df @ W.T
        assert_eq(dF_dx.shape, x.shape)
        
        # (features, output) = (features, samples) @ (samples, output)
        dF_dW = x.t() @ dF_df          
        assert_eq(dF_dW.shape, W.shape)
        
        # (output,) * (output,) -> (output, )
        dF_db = dF_df.sum(dim=0)     
        assert_eq(dF_db.shape, (FO,))
        
        return (dF_dx, dF_dW, dF_db)
    
#
# `Linear` derived from `nn.Module` is just a wrapper over `LinearFunction` for nicer API.
#
# For simplicity the `weight` is in shape of (in_features, out_features), 
# which is not consistent with the `nn.Linear`.
#
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, init: float = None):
        """Linear layer: y = x @ W + b"""

        super().__init__()

        if init is None:
            self.weight = nn.Parameter(tr.randn(in_features, out_features) * 0.01)
            self.bias = nn.Parameter(tr.randn(out_features, ))
        else:
            self.weight = nn.Parameter(tr.ones(in_features, out_features) * init)
            self.bias = nn.Parameter(tr.ones(out_features,) * init)

    def forward(self, x):
        return LinearFunction.apply(x, self.weight, self.bias)
    

def test_linear_function() -> None:
    tr.manual_seed(0)

    samples = 10
    in_features = 3
    out_features = 4

    x = tr.randn(samples, in_features, dtype=tr.float32, requires_grad=True)
    W = tr.randn(in_features, out_features, dtype=tr.float32, requires_grad=True)
    b = tr.randn(out_features, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    W1 = W.clone().detach().requires_grad_(True)
    b1 = b.clone().detach().requires_grad_(True)
    F1 = linear(x1, W1, b1).sum()
    F1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    W2 = W.clone().detach().requires_grad_(True)
    b2 = b.clone().detach().requires_grad_(True)
    F2 = (x2 @ W2 + b2).sum()
    F2.backward()

    assert_eq(x1, x2, atol=0.001)
    assert_eq(x1.grad, x2.grad, atol=0.001)
    assert_eq(W1, W2, atol=0.001)
    assert_eq(W1.grad, W2.grad, atol=0.001)
    assert_eq(b1, b2, atol=0.001)
    assert_eq(b1.grad, b2.grad, atol=0.001)


def test_linear_module() -> None:
    tr.manual_seed(0)

    samples = 3
    in_features = 4
    out_features = 5

    x = tr.randn(samples, in_features, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    linear = Linear(in_features, out_features, init=0.5)
    F1 = linear(x1).sum()
    F1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    nn_linear = nn.Linear(in_features, out_features)
    with tr.no_grad():
        nn_linear.weight.copy_(tr.ones_like(nn_linear.weight) * 0.5)
        nn_linear.bias.copy_(tr.ones_like(nn_linear.bias) * 0.5)
    F2 = nn_linear(x2).sum()
    F2.backward()

    assert_eq(x1, x2, atol=0.001)
    assert_eq(x1.grad, x2.grad, atol=0.001)
    assert_eq(linear.weight.data, nn_linear.weight.data, atol=0.001)
    assert_eq(linear.weight.grad, nn_linear.weight.grad, atol=0.001)
    assert_eq(linear.bias.data, nn_linear.bias.data, atol=0.001)
    assert_eq(linear.bias.grad, nn_linear.bias.grad, atol=0.001)


if __name__ == "__main__":
    test_linear_function()
    test_linear_module()
